# Introduccion a Redes Neuronales
## Deep Learning
### Inteligencia Artificial: De los Fundamentos a la Implementacion

---

**Datasets usados en esta clase:**
- MNIST (digitos escritos a mano) -- disponible directamente en TensorFlow/Keras
- ventas.csv -- lo retomamos en clases posteriores para prediccion con redes

**Librerias requeridas:** numpy, matplotlib, tensorflow (incluye keras)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# TensorFlow es el framework de Deep Learning que usaremos
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Semilla para reproducibilidad: garantiza que todos obtengan los mismos resultados
np.random.seed(42)
tf.random.set_seed(42)

print('NumPy version:      ', np.__version__)
print('TensorFlow version: ', tf.__version__)
print()
print('Configuracion lista. Comenzamos!')

---
# El Perceptron
## La unidad basica de una red neuronal

---

### De la neurona biologica a la neurona artificial

Las redes neuronales artificiales se inspiran (de forma muy simplificada)
en como funciona el cerebro humano.

```
NEURONA BIOLOGICA          NEURONA ARTIFICIAL (Perceptron)
--------------------       --------------------------------

  Dendrita                   Entrada (x)
  Recibe senales             Recibe valores numericos

  Sinapsis                   Peso (w)
  Fuerza de conexion         Importancia de cada entrada

  Cuerpo celular             Suma ponderada + sesgo
  Integra senales            z = w1*x1 + w2*x2 + ... + b

  Axon                       Funcion de activacion
  Dispara o no               Decide si 'activa' la neurona

  Neurona siguiente          Salida (y)
  Recibe la senal            Pasa a la siguiente capa
```

### El Perceptron de Rosenblatt (1958)

El perceptron fue el primer modelo de neurona artificial. Es sencillo pero
es la base de todo lo que viene despues.

```
ESTRUCTURA DEL PERCEPTRON
--------------------------

  ENTRADAS     PESOS       SUMA PONDERADA    ACTIVACION   SALIDA

  x1 ------> (w1) ---+
                      |
  x2 ------> (w2) ---+--->  z = sum(wi*xi) + b  --->  f(z)  ---> y
                      |
  x3 ------> (w3) ---+
                      |
   1 ------> (b)  ---+  <- sesgo (bias)

  Donde:
    xi = valor de entrada i
    wi = peso asociado a la entrada i (lo que el modelo aprende)
    b  = sesgo (permite desplazar la activacion)
    z  = suma ponderada (activacion lineal)
    f  = funcion de activacion (introduce no-linealidad)
    y  = salida de la neurona
```

### Matematicamente:

```
  z = w1*x1 + w2*x2 + ... + wn*xn + b
  z = W . X + b          (notacion vectorial)
  y = f(z)               (aplicamos la funcion de activacion)
```

### Como aprende el perceptron:

```
  1. Inicializar pesos w al azar (pequenos valores)
  2. Para cada ejemplo de entrenamiento:
     a. Calcular prediccion:   y_pred = f(W . X + b)
     b. Calcular error:        error = y_real - y_pred
     c. Actualizar pesos:      w = w + lr * error * x
     d. Actualizar sesgo:      b = b + lr * error
  3. Repetir hasta convergencia

  lr = learning rate (tasa de aprendizaje)
```

### Limitacion del perceptron simple:

```
  Puede resolver:          No puede resolver:

  AND, OR                  XOR (exclusivo)
  (linealmente             (no es linealmente
   separables)              separable)

  o  |  o              o  |  x
  o  |  x              x  |  o
  ---+---              ---+---
  (una linea separa)   (ninguna linea separa)

  Solucion: apilar multiples capas de perceptrones
            -> Red Neuronal Multicapa (MLP)
```

---
# Arquitectura de Redes Neuronales
## Perceptron a la red multicapa

---

### Red Neuronal Multicapa (MLP -- Multi-Layer Perceptron)

Una red neuronal es simplemente un conjunto de perceptrones organizados en capas.
Cada neurona en una capa recibe las salidas de TODAS las neuronas de la capa anterior.

```
ARQUITECTURA DE UNA RED NEURONAL TIPICA
----------------------------------------

  CAPA DE ENTRADA    CAPA OCULTA 1    CAPA OCULTA 2    CAPA DE SALIDA
  (Input Layer)      (Hidden Layer)   (Hidden Layer)   (Output Layer)

      [x1]  o                                              o  [y1]
             \ o   o /                  o   o              
      [x2]  o-X   X--o-o-o-o-o-o-o-o--X   X-o-----------o  [y2]
             / o   o \                  o   o
      [x3]  o   o   o                  o   o              o  [y3]
             \ o   o /                  o   o
      [x4]  o-X   X--o-o-o-o-o-o-o-o--X   X
             / o   o \                  o   o
      [x5]  o                                              

  Cada 'o' es una neurona (perceptron)
  Cada linea es una conexion con un peso w
  Las conexiones van en una sola direccion: izquierda a derecha
```

### Las 3 tipos de capas:

```
1. CAPA DE ENTRADA (Input Layer)
   - Una neurona por cada feature del dataset
   - No hace calculos: solo recibe y pasa los datos
   - Ejemplo: imagen 28x28 pixeles = 784 neuronas de entrada

2. CAPAS OCULTAS (Hidden Layers)
   - Aqui ocurre el 'aprendizaje'
   - Cada capa aprende representaciones mas abstractas
   - El numero de capas y neuronas es un hiperparametro
   - Capa 1: aprende bordes y lineas
   - Capa 2: aprende formas simples
   - Capa 3: aprende conceptos complejos

3. CAPA DE SALIDA (Output Layer)
   - Una neurona por clase (clasificacion multiclase)
   - Una neurona (clasificacion binaria o regresion)
   - Ejemplo: clasificar 10 digitos = 10 neuronas de salida
```

### Terminologia importante:

```
  TERMINO         SIGNIFICADO
  -----------     --------------------------------------------------
  Profundidad     Numero de capas ocultas
  Ancho           Numero de neuronas por capa
  Parametros      Total de pesos (w) y sesgos (b) que aprende el modelo
  Epoch           Una pasada completa por todos los datos de entrenamiento
  Batch           Subconjunto de datos usado en cada actualizacion de pesos
  Dense/FC        Capa densa: cada neurona conecta con TODAS las anteriores

  Ejemplo de conteo de parametros:
  Red: [784] -> [128] -> [64] -> [10]

  Capa 1: 784 entradas * 128 neuronas + 128 sesgos = 100,480 parametros
  Capa 2: 128 entradas *  64 neuronas +  64 sesgos =   8,256 parametros
  Capa 3:  64 entradas *  10 neuronas +  10 sesgos =     650 parametros
  TOTAL:                                             = 109,386 parametros
```

### Por que necesitamos capas ocultas?

```
  Sin capas ocultas -> Solo puede aprender funciones lineales
  Con capas ocultas -> Puede aprender funciones arbitrariamente complejas

  Teorema de aproximacion universal:
  Una red con una sola capa oculta y suficientes neuronas puede aproximar
  CUALQUIER funcion continua. En practica, redes mas profundas aprenden
  mejor con menos neuronas.
```

---
# Funciones de Activacion
## La fuente de la no-linealidad

---

### Por que necesitamos funciones de activacion?

```
  Sin activacion no-lineal:
  capa1 = W1 * X  +  b1
  capa2 = W2 * capa1  +  b2  =  W2*(W1*X + b1) + b2  =  W_equiv * X + b_equiv

  Apilando capas lineales solo obtenemos OTRA funcion lineal.
  Podria tener 100 capas y el resultado es equivalente a 1 capa.

  Con activacion no-lineal:
  capa1 = f1(W1 * X + b1)
  capa2 = f2(W2 * capa1 + b2)  <- ya no es equivalente a 1 capa

  La no-linealidad permite a la red aprender patrones complejos.
```

### Las funciones de activacion mas importantes:

```
  1. SIGMOID (sigmoide)
  ----------------------
  formula:  f(z) = 1 / (1 + e^(-z))
  rango:    (0, 1)
  uso:      salida de clasificacion binaria

  z:   ...-3  -2  -1   0   1   2   3...
  f:   ...0.05 0.12 0.27 0.5 0.73 0.88 0.95...

  Problema: 'vanishing gradient' en valores extremos (la curva se aplana)

  2. TANH (tangente hiperbolica)
  --------------------------------
  formula:  f(z) = (e^z - e^(-z)) / (e^z + e^(-z))
  rango:    (-1, 1)
  uso:      capas ocultas (mejor que sigmoid, centrada en 0)

  z:   ...-3  -2  -1   0   1   2   3...
  f:   ...-0.99 -0.96 -0.76 0 0.76 0.96 0.99...

  3. ReLU (Rectified Linear Unit) -- LA MAS USADA HOY EN DIA
  -----------------------------------------------------------
  formula:  f(z) = max(0, z)
  rango:    [0, +infinito)
  uso:      capas ocultas en casi todas las arquitecturas modernas

  z:   ...-3  -2  -1   0   1   2   3...
  f:   ... 0   0   0   0   1   2   3...

  Ventajas: simple, rapida, sin vanishing gradient en la parte positiva
  Problema: 'dying ReLU' (neuronas que quedan en 0 permanentemente)

  4. Leaky ReLU
  ---------------
  formula:  f(z) = max(0.01*z, z)
  uso:      alternativa a ReLU que evita el dying ReLU

  z:   ...-3  -2  -1   0   1   2   3...
  f:   ...-0.03 -0.02 -0.01 0   1   2   3...

  5. SOFTMAX
  -----------
  formula:  f(zi) = e^zi / sum(e^zj)  para cada clase j
  rango:    (0, 1), todas las salidas suman 1
  uso:      capa de salida para clasificacion multiclase

  Convierte un vector de scores en probabilidades:
  [2.0, 1.0, 0.1]  ->  [0.66, 0.24, 0.10]
  La clase con mayor probabilidad es la prediccion
```

### Como elegir la funcion de activacion:

```
  CAPA OCULTA:    ReLU es el punto de partida. Leaky ReLU si hay dying ReLU.
  SALIDA BINARIA: Sigmoid  (da probabilidad entre 0 y 1)
  SALIDA MULTI:   Softmax  (da probabilidades que suman 1)
  SALIDA REGRES:  Ninguna / Lineal (queremos valores reales sin restriccion)
```

### Visualizacion de las funciones de activacion

In [ ]:
# ---------------------------------------------------------------
# VISUALIZACION: Funciones de Activacion
# Graficamos las 4 funciones principales para ver su comportamiento
# ---------------------------------------------------------------

z = np.linspace(-5, 5, 300)  # Rango de valores de entrada

# Definimos cada funcion de activacion manualmente para que quede claro

def sigmoid(z):
    # Aplana la salida entre 0 y 1
    return 1 / (1 + np.exp(-z))

def tanh_fn(z):
    # Similar a sigmoid pero centrada en 0 (rango -1 a 1)
    return np.tanh(z)

def relu(z):
    # Muy simple: si es positivo lo deja, si es negativo lo hace 0
    return np.maximum(0, z)

def leaky_relu(z, alpha=0.1):
    # Como ReLU pero con una pendiente pequeña en la parte negativa
    return np.where(z > 0, z, alpha * z)

# Creamos la figura con 4 subplots, uno por funcion
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Funciones de Activacion en Redes Neuronales', fontsize=14)

# Colores de la paleta del curso
colores = ['#0f62fe', '#009d9a', '#8a3ffc', '#ff7eb6']

activaciones = [
    ('Sigmoid  f(z) = 1/(1+e^-z)',   sigmoid(z),      'Salida binaria'),
    ('Tanh     f(z) = tanh(z)',       tanh_fn(z),      'Capas ocultas (centrada en 0)'),
    ('ReLU     f(z) = max(0, z)',     relu(z),         'Capas ocultas (la mas usada)'),
    ('Leaky ReLU  f(z) = max(0.1z, z)', leaky_relu(z), 'Alternativa a ReLU'),
]

for ax, (titulo, valores, uso), color in zip(axes.flatten(), activaciones, colores):
    ax.plot(z, valores, color=color, linewidth=2.5)
    ax.axhline(y=0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.axvline(x=0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_title(titulo, fontsize=11, pad=8)
    ax.set_xlabel('z (entrada)')
    ax.set_ylabel('f(z) (salida)')
    ax.grid(True, alpha=0.3)
    # Anotamos el uso tipico de cada funcion
    ax.text(0.05, 0.05, f'Uso: {uso}', transform=ax.transAxes,
            fontsize=8, color='#555', bbox=dict(boxstyle='round', facecolor='#eee', alpha=0.8))

plt.tight_layout()
plt.show()

print('Observar:')
print('  - Sigmoid y Tanh se aplanan en los extremos (problema de vanishing gradient)')
print('  - ReLU es lineal en la parte positiva (sin aplananamiento)')
print('  - Leaky ReLU no aplana nunca en ninguna direccion')

---
#Forward Pass y Backpropagation
## Como aprende una red neuronal

---

### El ciclo de aprendizaje

Una red neuronal aprende repitiendo siempre el mismo ciclo de 4 pasos:

```
  +----------------------------------------------------------+
  |                  CICLO DE APRENDIZAJE                   |
  +----------------------------------------------------------+
  |
  |   PASO 1: FORWARD PASS
  |   Los datos viajan de izquierda a derecha a traves de la red
  |   Entrada -> Capa 1 -> Capa 2 -> ... -> Salida (prediccion)
  |
  |   PASO 2: CALCULAR PERDIDA (Loss)
  |   Comparamos la prediccion con el valor real
  |   Loss = funcion que mide cuan equivocado esta el modelo
  |
  |   PASO 3: BACKWARD PASS (Backpropagation)
  |   La senal de error viaja de derecha a izquierda
  |   Calcula cuanto contribuyo cada peso al error
  |   Usa la regla de la cadena del calculo diferencial
  |
  |   PASO 4: ACTUALIZAR PESOS (Gradient Descent)
  |   Ajustamos todos los pesos en la direccion que reduce el error
  |   w = w - lr * (gradiente)
  |
  +----------------------------------------------------------+

     DATOS --> [RED] --> PREDICCION
                              |
                           vs REAL
                              |
                           PERDIDA
                              |
               [BACKPROP] <---+
               calcula gradientes
                              |
               [ACTUALIZA PESOS] <---+
               w = w - lr * grad
```

### Forward Pass en detalle:

```
  Para cada capa L:
    z[L] = W[L] * a[L-1] + b[L]   <- suma ponderada
    a[L] = f(z[L])                 <- aplicar activacion

  Donde:
    W[L] = matriz de pesos de la capa L
    b[L] = vector de sesgos de la capa L
    a[L] = activaciones (salidas) de la capa L
    a[0] = X (las entradas originales)
```

### Funciones de Perdida (Loss):

```
  PROBLEMA               FUNCION DE PERDIDA         FORMULA SIMPLIFICADA
  --------------------   ------------------------   ---------------------
  Regresion              MSE (Error cuadratico)     mean((y - y_pred)^2)
  Clasificacion binaria  Binary Crossentropy        -mean(y*log(p) + (1-y)*log(1-p))
  Clasificacion multi    Categorical Crossentropy   -mean(sum(y * log(p)))

  Intuitivamente: la perdida mide 'que tan lejos estamos'.
  El objetivo del entrenamiento es MINIMIZAR la perdida.
```

### Backpropagation -- La idea central:

```
  Backpropagation = calcular gradientes usando la regla de la cadena

  La regla de la cadena dice:
  Si y = f(g(x)), entonces dy/dx = (dy/dg) * (dg/dx)

  En la red:
  dL/dW[L] = dL/da[L] * da[L]/dz[L] * dz[L]/dW[L]

  Esto se calcula de atras hacia adelante (de la salida a la entrada)
  y es lo que hace TensorFlow automaticamente con 'autodiferenciacion'.

  Nosotros NO necesitamos calcular esto a mano.
  TensorFlow lo hace por nosotros. Pero es importante entender la idea.
```

### Optimizadores -- Como actualizamos los pesos:

```
  OPTIMIZADOR       DESCRIPCION                              CUÁNDO USAR
  ----------------  ---------------------------------------  -----------
  SGD               Descenso de gradiente basico             Referencia
  SGD + Momentum    SGD que acumula velocidad               Mejor que SGD solo
  Adam              Adapta el lr por parametro + momentum   La mejor opcion por defecto
  RMSprop           Adapta el lr, bueno para series templ   Series temporales

  Para empezar: usar siempre Adam. Funciona bien en casi todos los casos.
```

---
# PRACTICA 1 -- Perceptron desde cero en NumPy

Implementamos un perceptron manualmente para entender exactamente lo que hace.
Sin librerias de deep learning -- solo NumPy.

In [ ]:
# ---------------------------------------------------------------
# PRACTICA 1: Perceptron desde cero
# Implementamos el algoritmo del perceptron original (Rosenblatt)
# para clasificacion binaria
# ---------------------------------------------------------------

class Perceptron:
    """
    Implementacion del perceptron original de Rosenblatt (1958).
    Clasifica datos linealmente separables en dos categorias (0 o 1).
    """

    def __init__(self, n_entradas, lr=0.1, n_epochs=100):
        """
        Inicializa el perceptron.

        Parametros:
          n_entradas: cuantas features tiene el dataset
          lr:         learning rate (tasa de aprendizaje)
          n_epochs:   cuantas veces pasamos por todos los datos
        """
        # Los pesos se inicializan en 0 (o valores pequeños aleatorios)
        # Un peso por cada feature de entrada
        self.w = np.zeros(n_entradas)

        # El sesgo (bias) tambien empieza en 0
        self.b = 0.0

        self.lr       = lr
        self.n_epochs = n_epochs

        # Guardamos el historial de errores para graficar el aprendizaje
        self.historial_errores = []

    def _activacion_escalon(self, z):
        """
        Funcion de activacion escalon (step function).
        Retorna 1 si z >= 0, y 0 si z < 0.
        Es la funcion de activacion del perceptron original.
        """
        return 1 if z >= 0 else 0

    def predecir_uno(self, x):
        """
        Hace la prediccion para UN solo ejemplo.
        Paso 1: suma ponderada    z = w . x + b
        Paso 2: aplica activacion y = f(z)
        """
        z = np.dot(self.w, x) + self.b   # Suma ponderada
        return self._activacion_escalon(z) # Activacion

    def predecir(self, X):
        """
        Hace predicciones para un conjunto completo de ejemplos.
        """
        return np.array([self.predecir_uno(x) for x in X])

    def entrenar(self, X, y):
        """
        Entrena el perceptron usando la regla de aprendizaje del perceptron.

        Para cada ejemplo mal clasificado:
          - Si predijo 0 pero era 1: aumentar pesos en la direccion de x
          - Si predijo 1 pero era 0: disminuir pesos en la direccion de x
        """
        for epoch in range(self.n_epochs):
            errores_epoch = 0

            # Iteramos por cada ejemplo del dataset
            for xi, yi_real in zip(X, y):

                # Prediccion actual del modelo
                yi_pred = self.predecir_uno(xi)

                # Calculamos el error: 0 si acerto, +1 o -1 si fallo
                error = yi_real - yi_pred

                # Regla de actualizacion del perceptron:
                # Si el error es 0 (acerto), los pesos no cambian
                # Si el error es !=0 (fallo), ajustamos los pesos
                self.w += self.lr * error * xi  # Actualizar pesos
                self.b += self.lr * error       # Actualizar sesgo

                # Contamos cuantos errores hubo en este epoch
                errores_epoch += int(error != 0)

            # Guardamos cuantos errores hubo en este epoch
            self.historial_errores.append(errores_epoch)

            # Si no hubo errores, el modelo convergio: ya no aprende mas
            if errores_epoch == 0:
                print(f'Convergencia en epoch {epoch + 1}')
                break

        return self


# ---------------------------------------------------------------
# Prueba 1: Problema AND (linealmente separable -> el perceptron puede resolverlo)
# ---------------------------------------------------------------
print('=== PROBLEMA AND ===')
print('Entradas: x1, x2 | Salida esperada: x1 AND x2')
print()

# Tabla de verdad AND
X_and = np.array([[0, 0],
                  [0, 1],
                  [1, 0],
                  [1, 1]])

y_and = np.array([0, 0, 0, 1])  # Solo (1,1) da 1

# Creamos y entrenamos el perceptron
p_and = Perceptron(n_entradas=2, lr=0.1, n_epochs=100)
p_and.entrenar(X_and, y_and)

# Evaluamos
predicciones_and = p_and.predecir(X_and)
exactitud_and = np.mean(predicciones_and == y_and) * 100

print(f'Pesos aprendidos:  w = {p_and.w}')
print(f'Sesgo aprendido:   b = {p_and.b:.3f}')
print(f'Exactitud:         {exactitud_and:.0f}%')
print()
print('Predicciones:')
for x, y_real, y_pred in zip(X_and, y_and, predicciones_and):
    estado = 'OK' if y_real == y_pred else 'MAL'
    print(f'  AND({x[0]}, {x[1]}) = {y_real}  ->  prediccion: {y_pred}  [{estado}]')

In [ ]:
# ---------------------------------------------------------------
# Prueba 2: Problema XOR (NO linealmente separable -> el perceptron falla)
# Esto motiva la necesidad de redes multicapa
# ---------------------------------------------------------------
print('=== PROBLEMA XOR ===')
print('Entradas: x1, x2 | Salida esperada: x1 XOR x2')
print('(XOR da 1 cuando exactamente UNA entrada es 1)')
print()

# Tabla de verdad XOR
X_xor = np.array([[0, 0],
                  [0, 1],
                  [1, 0],
                  [1, 1]])

y_xor = np.array([0, 1, 1, 0])  # XOR: diferentes = 1, iguales = 0

p_xor = Perceptron(n_entradas=2, lr=0.1, n_epochs=100)
p_xor.entrenar(X_xor, y_xor)

predicciones_xor = p_xor.predecir(X_xor)
exactitud_xor = np.mean(predicciones_xor == y_xor) * 100

print(f'Exactitud: {exactitud_xor:.0f}%  <- nunca llega al 100%')
print()
print('Predicciones:')
for x, y_real, y_pred in zip(X_xor, y_xor, predicciones_xor):
    estado = 'OK' if y_real == y_pred else 'MAL'
    print(f'  XOR({x[0]}, {x[1]}) = {y_real}  ->  prediccion: {y_pred}  [{estado}]')

print()
print('Conclusion: el perceptron simple NO puede resolver XOR.')
print('Necesitamos una red con al menos 1 capa oculta.')
print('Esto lo resolvemos con TensorFlow en la siguiente practica.')

# ---------------------------------------------------------------
# Graficamos el historial de errores durante el entrenamiento
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# AND: deberia converger rapidamente
axes[0].plot(p_and.historial_errores, color='#009d9a', linewidth=2)
axes[0].set_title('Problema AND -- Errores por Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cantidad de errores')
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.3)

# XOR: nunca converge al 0
axes[1].plot(p_xor.historial_errores, color='#8a3ffc', linewidth=2)
axes[1].set_title('Problema XOR -- Errores por Epoch (no converge)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Cantidad de errores')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# PRACTICA 2 -- Primera red neuronal con TensorFlow/Keras

Ahora usamos TensorFlow para construir una red multicapa.
Primero resolvemos el XOR (que el perceptron no pudo), y luego
pasamos al dataset real MNIST.

In [ ]:
# ---------------------------------------------------------------
# PRACTICA 2a: Resolviendo XOR con una red multicapa en Keras
# Demostramos que una capa oculta resuelve lo que el perceptron no puede
# ---------------------------------------------------------------

print('=== XOR con Red Multicapa en Keras ===')
print()

# Los mismos datos de XOR de antes
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype='float32')
y_xor = np.array([0, 1, 1, 0], dtype='float32')

# ---------------------------------------------------------------
# Construimos el modelo con la API Sequential de Keras
# Sequential = las capas se apilan una tras otra en secuencia
# ---------------------------------------------------------------
modelo_xor = keras.Sequential([

    # Capa de entrada: 2 neuronas (una por feature)
    # No la ponemos explicitamente, Keras la infiere del input_shape

    # Primera capa oculta: 4 neuronas con activacion ReLU
    # input_shape=(2,) dice que cada ejemplo tiene 2 features
    layers.Dense(4, activation='relu', input_shape=(2,)),

    # Capa de salida: 1 neurona con sigmoid (clasificacion binaria)
    # Sigmoid da un valor entre 0 y 1 (probabilidad)
    layers.Dense(1, activation='sigmoid')

], name='red_xor')

# ---------------------------------------------------------------
# Compilamos el modelo
# Aqui definimos:
#   optimizer: como actualizar los pesos (Adam es el mas recomendado)
#   loss:      funcion de perdida (binary_crossentropy para clasificacion binaria)
#   metrics:   que metricas mostrar durante el entrenamiento
# ---------------------------------------------------------------
modelo_xor.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Resumen del modelo: muestra la arquitectura y numero de parametros
modelo_xor.summary()

print()
print('Parametros totales del modelo:')
print('  Capa 1: 2 entradas * 4 neuronas + 4 sesgos = 12 parametros')
print('  Capa 2: 4 entradas * 1 neurona  + 1 sesgo  =  5 parametros')
print('  TOTAL:  17 parametros (todos se aprenden durante el entrenamiento)')

In [ ]:
# ---------------------------------------------------------------
# Entrenamos el modelo XOR
# epochs=1000: pasamos 1000 veces por los 4 ejemplos
# verbose=0:   no mostrar cada epoch (solo el resultado final)
# ---------------------------------------------------------------
historia_xor = modelo_xor.fit(
    X_xor, y_xor,
    epochs=1000,
    verbose=0   # Silencioso para no llenar la pantalla
)

# Evaluamos el modelo en los datos de entrenamiento
perdida, exactitud = modelo_xor.evaluate(X_xor, y_xor, verbose=0)
print(f'Perdida final:  {perdida:.4f}')
print(f'Exactitud:      {exactitud * 100:.1f}%')
print()

# Mostramos las predicciones
predicciones = modelo_xor.predict(X_xor, verbose=0)
print('Predicciones de la red neuronal para XOR:')
for x, y_real, pred in zip(X_xor, y_xor, predicciones):
    clase_predicha = 1 if pred[0] >= 0.5 else 0
    estado = 'OK' if clase_predicha == y_real else 'MAL'
    print(f'  XOR({int(x[0])}, {int(x[1])}) = {int(y_real)} | '
          f'probabilidad: {pred[0]:.3f} | prediccion: {clase_predicha} [{estado}]')

print()
print('La red multicapa SI puede resolver XOR.')
print('La capa oculta aprende representaciones internas que hacen el problema separable.')

# Grafica del aprendizaje
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(historia_xor.history['loss'], color='#0f62fe')
axes[0].set_title('Perdida (Loss) durante el entrenamiento')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(historia_xor.history['accuracy'], color='#009d9a')
axes[1].set_title('Exactitud (Accuracy) durante el entrenamiento')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# PRACTICA 3 -- Clasificacion de digitos MNIST

MNIST es el 'Hola Mundo' del Deep Learning.
Contiene 70,000 imagenes de digitos escritos a mano (0-9), cada una de 28x28 pixeles.
Construimos una red neuronal que los clasifique.

In [ ]:
# ---------------------------------------------------------------
# PASO 1: Cargar y explorar el dataset MNIST
# Keras lo descarga automaticamente la primera vez
# ---------------------------------------------------------------
print('Cargando dataset MNIST...')

# mnist.load_data() retorna los datos ya divididos en train y test
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print(f'Datos de entrenamiento: {X_train.shape}  (60000 imagenes de 28x28 pixeles)')
print(f'Etiquetas train:        {y_train.shape}')
print(f'Datos de prueba:        {X_test.shape}   (10000 imagenes)')
print(f'Etiquetas test:         {y_test.shape}')
print()
print(f'Rango de valores de pixel: {X_train.min()} a {X_train.max()}')
print(f'Clases posibles:          {np.unique(y_train)}')
print()

# Visualizamos algunos ejemplos para entender los datos
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle('Ejemplos del dataset MNIST (digitos escritos a mano)', fontsize=12)

for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'clase: {y_train[i]}')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# PASO 2: Preprocesar los datos
# Las redes neuronales funcionan mejor con datos normalizados
# ---------------------------------------------------------------

# Normalizacion: dividir por 255 para pasar de [0,255] a [0,1]
# Esto ayuda al entrenamiento porque los gradientes son mas estables
X_train_norm = X_train / 255.0
X_test_norm  = X_test  / 255.0

# Aplanar (flatten): cada imagen de 28x28 se convierte en un vector de 784 valores
# La capa Dense no entiende imagenes 2D, necesita vectores 1D
# Esto lo podemos hacer con una capa Flatten en el modelo, o con reshape aqui
X_train_flat = X_train_norm.reshape(-1, 28 * 28)  # -1 significa 'todas las imagenes'
X_test_flat  = X_test_norm.reshape(-1, 28 * 28)

print('Transformacion de imagenes:')
print(f'  Antes:  {X_train_norm.shape}  (60000 imagenes 2D de 28x28)')
print(f'  Despues: {X_train_flat.shape}  (60000 vectores de 784 valores)')
print()
print(f'Rango de valores ANTES de normalizar: {X_train.min()} - {X_train.max()}')
print(f'Rango de valores DESPUES:             {X_train_flat.min():.1f} - {X_train_flat.max():.1f}')

In [ ]:
# ---------------------------------------------------------------
# PASO 3: Construir la red neuronal para MNIST
# Arquitectura: 784 -> 128 -> 64 -> 10
# ---------------------------------------------------------------

modelo_mnist = keras.Sequential([

    # Capa de entrada + primera capa oculta
    # 784 = 28*28 pixeles (una neurona de entrada por pixel)
    # 128 neuronas ocultas con activacion ReLU
    layers.Dense(128, activation='relu', input_shape=(784,), name='capa_oculta_1'),

    # Segunda capa oculta: 64 neuronas con ReLU
    # Cada capa aprende representaciones mas abstractas
    layers.Dense(64, activation='relu', name='capa_oculta_2'),

    # Capa de salida: 10 neuronas (una por cada digito 0-9)
    # Softmax convierte los scores en probabilidades que suman 1
    # La neurona con mayor probabilidad es la clase predicha
    layers.Dense(10, activation='softmax', name='capa_salida')

], name='clasificador_mnist')

# Compilamos el modelo
modelo_mnist.compile(
    # Adam: el optimizador mas utilizado para empezar
    optimizer='adam',

    # sparse_categorical_crossentropy: para clasificacion con multiples clases
    # 'sparse' porque las etiquetas son enteros (0,1,...,9) no one-hot vectors
    loss='sparse_categorical_crossentropy',

    # La metrica que vemos durante el entrenamiento
    metrics=['accuracy']
)

# Mostramos la arquitectura completa
modelo_mnist.summary()

print()
print('Arquitectura visualizada:')
print()
print('  ENTRADA   CAPA_1     CAPA_2    SALIDA')
print('  [784]  -> [128 ReLU] -> [64 ReLU] -> [10 Softmax]')
print()
print('  Cada numero es la cantidad de neuronas en esa capa.')

In [ ]:
# ---------------------------------------------------------------
# PASO 4: Entrenar el modelo
# batch_size=32: procesar 32 imagenes a la vez antes de actualizar pesos
# epochs=10:     pasar 10 veces por todos los datos de entrenamiento
# validation_split=0.1: usar el 10% del train como validacion durante entrenamiento
# ---------------------------------------------------------------
print('Entrenando la red neuronal...')
print('(60,000 imagenes x 10 epochs = 600,000 ejemplos procesados)')
print()

historia_mnist = modelo_mnist.fit(
    X_train_flat,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,  # 10% para validacion: medir si estamos generalizando
    verbose=1
)

print()
print('Entrenamiento completado.')

In [ ]:
# ---------------------------------------------------------------
# PASO 5: Evaluar el modelo en datos que nunca vio
# Los datos de TEST son los que importan: son desconocidos para el modelo
# ---------------------------------------------------------------
perdida_test, exactitud_test = modelo_mnist.evaluate(X_test_flat, y_test, verbose=0)

print(f'Resultados en datos de TEST (nunca vistos):')
print(f'  Perdida (Loss):  {perdida_test:.4f}')
print(f'  Exactitud:       {exactitud_test * 100:.2f}%')
print()
print('Con solo 10 epochs y una red simple, ya superamos el 97% de exactitud.')
print('Un modelo de ML clasico tipico obtiene ~97% tambien, pero requiere mas trabajo manual.')

# ---------------------------------------------------------------
# Graficas de entrenamiento
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Curvas de Aprendizaje -- Red Neuronal MNIST', fontsize=13)

epochs_rango = range(1, len(historia_mnist.history['loss']) + 1)

# Grafica de perdida
axes[0].plot(epochs_rango, historia_mnist.history['loss'],
             label='Entrenamiento', color='#0f62fe', linewidth=2)
axes[0].plot(epochs_rango, historia_mnist.history['val_loss'],
             label='Validacion', color='#ff7eb6', linewidth=2, linestyle='--')
axes[0].set_title('Perdida por Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Grafica de exactitud
axes[1].plot(epochs_rango, historia_mnist.history['accuracy'],
             label='Entrenamiento', color='#009d9a', linewidth=2)
axes[1].plot(epochs_rango, historia_mnist.history['val_accuracy'],
             label='Validacion', color='#8a3ffc', linewidth=2, linestyle='--')
axes[1].set_title('Exactitud por Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.9, 1.0)  # Zoom en la zona relevante
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print()
print('Interpretar las curvas:')
print('  - Train y Validacion van juntas: el modelo NO esta haciendo overfitting')
print('  - Ambas curvas bajan/suben consistentemente: aprendizaje estable')
print('  - Si train mejora pero val empeora: overfitting (lo veremos en Clase 10)')

In [ ]:
# ---------------------------------------------------------------
# PASO 6: Visualizar predicciones individuales
# Vemos que predice el modelo en ejemplos concretos
# ---------------------------------------------------------------

# Hacemos predicciones en los primeros 16 ejemplos del test
predicciones_proba = modelo_mnist.predict(X_test_flat[:16], verbose=0)

# La clase predicha es la neurona con mayor probabilidad
predicciones_clase = np.argmax(predicciones_proba, axis=1)

# Visualizamos las imagenes con sus etiquetas reales y predichas
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('Predicciones del modelo en datos de TEST', fontsize=12)

for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_test[i], cmap='gray')

    real  = y_test[i]
    pred  = predicciones_clase[i]
    proba = predicciones_proba[i][pred]

    # Titulo verde si acerto, rojo si fallo
    color = 'green' if real == pred else 'red'
    ax.set_title(f'Real:{real} Pred:{pred}\n({proba*100:.0f}%)', color=color, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

print('Verde = prediccion correcta | Rojo = error del modelo')
print('El porcentaje indica la confianza del modelo en su prediccion.')

In [ ]:
# ---------------------------------------------------------------
# BONUS: Guardar el modelo entrenado
# Los modelos de Keras se pueden guardar y cargar en cualquier momento
# ---------------------------------------------------------------

# Guardar el modelo completo (arquitectura + pesos + optimizador)
modelo_mnist.save('modelo_mnist_clase9.h5')
print('Modelo guardado como: modelo_mnist_clase9.h5')

# Para cargar el modelo despues:
# modelo_cargado = keras.models.load_model('modelo_mnist_clase9.h5')
# predicciones = modelo_cargado.predict(nuevos_datos)

print()
print('Resumen de lo que construimos:')
print()
print('  Arquitectura: [784] -> [128 ReLU] -> [64 ReLU] -> [10 Softmax]')
print(f'  Parametros:   {modelo_mnist.count_params():,}')
print(f'  Exactitud:    {exactitud_test * 100:.2f}%  en 10,000 imagenes nunca vistas')
print(f'  Epochs:       10')
print()